# DFS Lineup Optimizer — Core Algorithm

This notebook documents the **integer linear programming (ILP) optimizer** used to select the
highest-projected DraftKings NFL Classic lineup within the $50,000 salary cap.

It is a **reference notebook**: it explains and exercises the production code, it does not
contain it.

**Nothing here is duplicated any more.** The DK↔projection matching cascade and the DK
scoring conversion live in `fantasy/dfs/dfs_matching.py`; the ILP itself lives in
`fantasy/dfs/lineup_optimizer.py`. This notebook and `dfs_pipeline.ipynb` both IMPORT them.
The ILP used to be copy-pasted into both notebooks, exactly as the matcher was before its
cross-position defect (§4) was found in one copy and not the other.


## 1 · Imports & Setup

We rely on three external libraries beyond the standard library:

| Library | Role |
|---------|------|
| `pandas` | Player pool table manipulation |
| `pulp` | Open-source ILP solver (CBC backend) |
| `difflib` | Fuzzy name matching between DK and our model |

`pathlib.Path` is used throughout so the notebook works regardless of working directory.


In [ ]:
import io
import re
import sys
from pathlib import Path

import pandas as pd
import pulp

# Resolve fantasy/dfs (for dfs_matching) and fantasy_projections relative to the CWD.
# Works whether you run from the project root or from inside fantasy/dfs/.
_HERE = Path().resolve()
DFS_DIR = next(
    (p for p in [_HERE, _HERE / "fantasy" / "dfs", _HERE.parent / "dfs",
                 _HERE.parent.parent / "fantasy" / "dfs"]
     if (p / "dfs_matching.py").exists()),
    None,
)
if DFS_DIR is None:
    raise RuntimeError("Could not find dfs_matching.py (expected in fantasy/dfs/).")
if str(DFS_DIR) not in sys.path:
    sys.path.insert(0, str(DFS_DIR))

PROJ_DIR = next(
    (p for p in [
        _HERE / "fantasy_projections",
        _HERE.parent / "fantasy_projections",
        _HERE.parent.parent / "fantasy" / "fantasy_projections",
    ] if p.exists()),
    _HERE / "fantasy_projections",
)
if not PROJ_DIR.exists():
    raise RuntimeError(f"Could not find fantasy_projections directory. Searched multiple locations. PROJ_DIR={PROJ_DIR}")

BUDGET = 50_000

print(f"dfs_matching module : {DFS_DIR}")
print(f"Projections directory: {PROJ_DIR}")
print(f"Exists: {PROJ_DIR.exists()}")


## 2 · Normalisation — the three constrained axes

DraftKings and nflverse disagree on all three of the fields we match on, so each gets
its own normaliser in `dfs_matching.py`:

- **Name** — `norm_name()` lowercases, drops apostrophes/periods/commas, turns hyphens
  into spaces, and strips generational suffixes: `"Amon-Ra St. Brown"` and
  `"Amon Ra St Brown"` both become `amon ra st brown`; `"T.J. Hockenson"` and
  `"TJ Hockenson"` both become `tj hockenson`.
- **Team** — `norm_team()` maps DK/PFR abbreviations onto the nflverse codes the
  projection files use (`LAR→LA`, `JAC→JAX`, `WSH→WAS`, `OAK→LV`, `SD→LAC`, …).
- **Position** — `pos_class()` collapses `RB`/`FB`/`HB` into one backfield class and
  leaves every other position on its own. This is the single constraint that makes
  `Josh Allen (QB)` → `Josh Palmer (WR)` impossible.

Normalisation alone is **not** the match. It only makes the keys comparable; the
cascade in §4 decides.


In [ ]:
from dfs_matching import (
    ALIAS_GROUPS, ALL_STATUSES, FUZZY_MARGIN, FUZZY_MIN, MODEL_STATUSES,
    assert_objective_finite, calc_dk_proj_pts, format_match_report,
    match_row, match_status_counts, merge_projections,
    norm_name, norm_team, num, pos_class, ProjectionIndex,
)

# Quick sanity check — the three axes the cascade constrains on.
assert norm_name("De'Von Achane Jr.") == "devon achane"
assert norm_name("Patrick Mahomes II") == "patrick mahomes"
assert norm_name("T.J. Hockenson") == norm_name("TJ Hockenson") == "tj hockenson"
assert norm_name("Amon-Ra St. Brown") == "amon ra st brown"
assert norm_team("LAR") == norm_team("LA")          # DK vs nflverse abbreviation
assert norm_team("JAC") == norm_team("JAX")
assert pos_class("QB") != pos_class("WR")           # Josh Allen is never Josh Palmer
assert pos_class("FB") == pos_class("RB")           # the one compatibility class
print("Name / team / position normalisation OK")
print(f"fuzzy thresholds: score >= {FUZZY_MIN}, margin over runner-up >= {FUZZY_MARGIN}")
print(f"reviewed alias groups: {len(ALIAS_GROUPS)}")


## 3 · Projection & Salary Data Loading

### 3a — Our model projections

`load_projections()` reads the weekly CSV produced by `predict_fantasy.ipynb`.
These files live in `fantasy/fantasy_projections/projections_{season}_week{week:02d}.csv`
and contain our XGBoost-based `projected_pts` for every active skill-position player.

We use our model projections (rather than DK's season average) because:
- Our model incorporates current injury status, depth chart position, and matchup difficulty.
- DK's `AvgPointsPerGame` is a season-long average and doesn't react to week-specific context.
- In backtesting our model beats the 3-week rolling average baseline at every position.

### 3b — DraftKings salary export

`load_dk_salaries()` parses the CSV exported directly from the DK contest lobby.
The standard columns are `Position`, `Name`, `Salary`, `TeamAbbrev`, `AvgPointsPerGame`.
Salary may include a `$` prefix depending on the DK export version.

For **DST** (team defenses) we have no position-specific projection model, so we fall back to
DK's `AvgPointsPerGame` as a proxy. This is an area for future improvement.


In [ ]:
_DK_COL_MAP = {
    "Position":        "position",
    "Name":            "name",
    "Salary":          "salary",
    "TeamAbbrev":      "team",
    "AvgPointsPerGame":"avg_pts",
    "Game Info":       "game_info",
}

# DK's own "ID" column is a DraftKings id, NOT a gsis id — mapping it would match
# nothing. Only a real nflverse id column enables step 1 of the cascade.
_DK_ID_COL_MAP = {"gsis_id": "player_id", "nfl_id": "player_id", "player_id": "player_id"}


def available_weeks() -> list[tuple[int, int]]:
    """Return (season, week) tuples for existing projection CSVs, newest first."""
    files = sorted(PROJ_DIR.glob("projections_*_week*.csv"), reverse=True)
    out = []
    for f in files:
        m = re.match(r"projections_(\d{4})_week(\d{2})\.csv", f.name)
        if m:
            out.append((int(m.group(1)), int(m.group(2))))
    return out


def load_projections(season: int, week: int) -> pd.DataFrame:
    path = PROJ_DIR / f"projections_{season}_week{week:02d}.csv"
    if not path.exists():
        raise FileNotFoundError(f"No projection file: {path}")
    return pd.read_csv(path)


def load_dk_salaries(path_or_bytes) -> pd.DataFrame:
    """Accept a file path (str/Path) or raw bytes from an uploaded file."""
    if isinstance(path_or_bytes, (str, Path)):
        raw = open(path_or_bytes, "rb").read()
    else:
        raw = path_or_bytes

    df = pd.read_csv(io.BytesIO(raw))
    df = df.rename(columns={k: v for k, v in _DK_COL_MAP.items() if k in df.columns})
    df = df.rename(columns={k: v for k, v in _DK_ID_COL_MAP.items() if k in df.columns})

    df["salary"] = (
        df["salary"].astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
    )
    df["salary"]  = pd.to_numeric(df["salary"],  errors="coerce")
    df["avg_pts"] = pd.to_numeric(df.get("avg_pts", 0), errors="coerce").fillna(0)
    df = df.dropna(subset=["salary", "name", "position"])

    keep = ["position", "name", "salary", "team", "avg_pts", "game_info", "player_id"]
    return df[[c for c in keep if c in df.columns]].copy()


weeks = available_weeks()
print(f"Available projection weeks: {weeks[:5]}")


## 4 · Merging Projections with Salaries — the constrained cascade

**This section used to be a one-line fuzzy match and it was wrong.** The old
`merge_projections()` called `difflib.get_close_matches(cutoff=0.72)` against *every*
projection name — no position filter, no team filter. Since `predict_fantasy` drops
every player ruled Out, a real DK export always contains names the projection file
lacks, so the fallback fired constantly. Leave-one-out over the 568 names in
`projections_2025_week10.csv`: **150 (26.4%) of absent players still got a "match", 95
of them at a different position**, and all of them were labelled `model`:

| DK row | matched to |
|---|---|
| Josh Allen (QB, BUF) | Josh Palmer (WR, BUF) |
| Aaron Rodgers (QB, PIT) | Aaron Jones (RB, MIN) |
| Mac Jones (QB, SF) | Zay Jones (WR, ARI) |
| Caleb Williams (QB, CHI) | Kyle Williams (WR, NE) |

`dfs_matching.merge_projections()` replaces it with a cascade, first hit wins:

| # | Status | Rule |
|---|---|---|
| 1 | `id` | stable player id equal on both sides |
| 2 | `exact` | normalized name + compatible position + same team, exactly one candidate |
| 3 | `team_mismatch` | name+position unique league-wide, teams disagree (trade / stale team) — projection used, row flagged |
| 4 | `alias` | hand-reviewed spelling table, still position-constrained |
| 5 | `fuzzy` | same team **and** same position pool only; needs score ≥ `FUZZY_MIN` **and** a `FUZZY_MARGIN` gap over the runner-up |
| — | `ambiguous` | ≥2 survivors, or the fuzzy leader missed the margin. Never guessed |
| — | `unmatched` | nothing plausible |
| — | `dst` | team defense; no DST model |

Rows that did not match still enter the pool on DK's season average (the optimizer needs
enough bodies to fill every slot) — but they carry their own status, so the pipeline
notebook can list them. **There is no generic `model` label any more**; use `used_model`.

Leave-one-out false-match rate after the fix: **0/568**. Thresholds and their derivation
are documented at the top of `dfs_matching.py` — the team+position pool does the work,
the score/margin pair is defence in depth.

The **value** column is `proj_pts / (salary / 1000)` — points per $1,000 of salary.


In [ ]:
# merge_projections now lives in dfs_matching.py (shared by dfs_pipeline.ipynb).
# Reproduce the defect and the fix on the real week-10 file:
from dfs_matching import loo_match_rate

_demo_weeks = available_weeks()
if _demo_weeks:
    _s, _w = _demo_weeks[0]
    _proj = load_projections(_s, _w)
    for _m in ("legacy", "cascade"):
        _r = loo_match_rate(_proj, matcher=_m)
        print(f"{_m:>8}: {_r['false_matches']}/{_r['n']} absent players falsely matched "
              f"({_r['rate']:.1%}), {_r['cross_position']} cross-position")
    print("\nA few legacy substitutions:")
    for a, b, _ in loo_match_rate(_proj, matcher="legacy")["examples"][:6]:
        print(f"  {a:<34} -> {b}")
else:
    print("No projection files present; skipping the demo.")


## 5 · Integer Linear Programming Formulation

The DK Classic lineup problem is a **binary integer program**, implemented once in
`lineup_optimizer.optimize_lineup()`.

**Decision variables:**  $x_i \in \{0, 1\}$ for each player $i$ in the pool.

**Objective:** Maximise total projected points
$$\max \sum_i \text{proj\_pts}_i \cdot x_i$$

**Constraints:**

| Rule | Formula |
|------|---------|
| Salary cap | $\sum_i \text{salary}_i \cdot x_i \leq 50{,}000$ |
| Total players | $\sum_i x_i = 9$ |
| QB | $\sum_{i \in QB} x_i = 1$ |
| RB | $\sum_{i \in RB} x_i \geq 2$ |
| WR | $\sum_{i \in WR} x_i \geq 3$ |
| TE | $\sum_{i \in TE} x_i \geq 1$ |
| DST | $\sum_{i \in DST} x_i = 1$ |
| Team max | $\sum_{i \in \text{team } t} x_i \leq 8 \quad \forall t$ |

The **FLEX slot** is implicit: with 9 total players, 1 QB, 1 DST, and the position minimums
(2 RB + 3 WR + 1 TE = 6), one extra RB, WR, or TE is chosen automatically by the solver
to fill the 9th slot — i.e. RB+WR+TE always sums to 7.

The solver is **PuLP's bundled CBC** (`pulp.PULP_CBC_CMD`) — a real branch-and-bound MILP
binary, never a continuous-relaxation or scipy stand-in. `assert_cbc_available()` below
records the binary that is actually installed; `test_lineup_optimizer.py` fails (it does not
skip) if CBC is missing. Lock/exclude constraints add equality bounds directly to the LP,
and a locked name that is not in the pool raises rather than silently doing nothing.


In [ ]:
# The ILP is production code, not a notebook cell: fantasy/dfs/lineup_optimizer.py.
from lineup_optimizer import (
    BUDGET, assert_cbc_available, assign_slots, lineup_facts, optimize_lineup,
)

# Record the solver that will actually run — measured, not assumed.
for _k, _v in assert_cbc_available().items():
    print(f"{_k:16s}: {_v}")


## 6 · Interpreting Results

**Value metric** (`proj_pts / (salary / 1000)`)
- A value of 4.0 means the player is projected to score 4 pts per $1k of salary.
- The optimizer maximises total points — not value directly — but high-value players
  naturally unlock salary for a strong anchor pick (e.g. a $9k QB).

**FLEX pick logic**
- The ILP chooses the FLEX player globally — it may add a 3rd RB, 4th WR, or 2nd TE
  depending on which option adds the most projected points within the remaining cap.
- You don't need to pre-specify the FLEX position.

**When the optimizer returns `None`**
- Not enough players at a position (e.g. only 1 QB in the salary CSV).
- Lock constraints leave no feasible solution within the cap.
- Usually fixed by removing a lock or checking the uploaded CSV for missing rows.

## 7 · Next Steps

The current optimizer selects a single optimal lineup. Future improvements:

1. **Multi-lineup generation** — produce N distinct lineups for GPP tournaments using
   ownership diversity constraints (force different FLEX picks across lineups).
2. **Correlation / game-stacking** — add constraints to include 2+ players from the same game,
   exploiting score correlation in shootouts.
3. **Ownership leverage** — weight by inverse projected ownership to differentiate from the field.
4. **DST projection model** — train a simple model on defensive matchup difficulty, implied
   team total, and home/away to replace the DK season-average fallback.
5. **Salary efficiency bands** — flag players whose salary has moved since the season average
   was set, indicating recency-priced information the model may not yet reflect.
